# Harness test

Kernel: **Colab, T4 GPU**. Run all cells top to bottom.
If you re-run after a change, **restart the kernel first** (`common.data` only reads `COMP433_DATA` once, at import).

In [ ]:
# 1. Get the code onto the Colab machine
import os
import shutil
from getpass import getpass

REPO = "/content/repo"
BRANCH = "massimo/harness-v1"

if not os.path.exists(REPO):
    token = getpass("GitHub token: ")
    !git clone -q -b {BRANCH} https://{token}@github.com/MassPaol/comp433-final-project.git {REPO}
    del token
else:
    !git -C {REPO} pull -q

%cd {REPO}
!pip install -q -e . -r requirements.txt

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. Copy CIFAR-10 from the team folder to the Colab machine (faster than reading from Drive)
DRIVE_DATA = "/content/drive/MyDrive/comp433_cache/data"  # needs the comp433_cache shortcut in My Drive
LOCAL_DATA = "/content/data"

if not os.path.exists(f"{LOCAL_DATA}/cifar-10-python.tar.gz"):
    os.makedirs(LOCAL_DATA, exist_ok=True)
    shutil.copy(f"{DRIVE_DATA}/cifar-10-python.tar.gz", LOCAL_DATA)  # torchvision unpacks it on first load

os.environ["COMP433_DATA"] = LOCAL_DATA

In [ ]:
# 4. Import the project code (must come after COMP433_DATA is set) and check the data loads
import torch
from common import harness, data

assert data.DATA_ROOT == os.environ["COMP433_DATA"], (
    "common.data was imported before COMP433_DATA was set. Restart the kernel and run all cells again."
)
print("GPU:", torch.cuda.get_device_name() if torch.cuda.is_available() else "none (running on CPU)")

X, y, Xv, yv, classes = data.load_instance(0)
print(X.shape, Xv.shape, [data.CLASS_NAMES[c] for c in classes])  # expect (100, 32, 32, 3) (700, 32, 32, 3)

In [ ]:
# 5. Test the harness with a dummy model: expect val_acc = 0.5 on every seed
import numpy as np
from common.method import Method

class AlwaysZero(Method):
    def fit(self, X, y): return self
    def predict(self, X): return np.zeros(len(X), dtype=int)

TEST_DIR = "/content/test_results"
shutil.rmtree(TEST_DIR, ignore_errors=True)  # start fresh so re-runs don't pile up rows

rows = harness.run(AlwaysZero, data.TUNE_SEEDS[:2], challenge="c1", person="massimo",
                   method_name="always_zero", config_id="test", results_dir=TEST_DIR)
print(open(f"{TEST_DIR}/massimo.csv").read())